### Imports

In [2]:
import os
import json
import tempfile
import subprocess
from itertools import islice

import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download


### Template Code

In [3]:
#NOTE: YOU WILL NEED TO MAKE SURE YOU HAVE THE FILE "credentials.json" IN ORDER TO ACCESS THE API
import io
from googleapiclient.http import MediaIoBaseDownload



from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle
import os
import io # Required for downloading file content
import mimetypes # Required for guessing MIME type during upload
#NOTE: You might have to do pip install on a few google libraries before you can use this code
# Define the permissions scope
# 'drive.file' scope grants access only to files created or opened by this app.
SCOPES = ['https://www.googleapis.com/auth/drive']

def get_drive_service():
    """Authenticates the user and returns the Google Drive API service object."""
    creds = None
    # 1. Load existing credentials (token.pickle)
    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)
            
    # 2. If no valid creds, refresh or perform OAuth login
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # Refresh expired token silently
            creds.refresh(Request())
        else:
            # Perform full login flow, requires 'credentials.json' in the same directory
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
            
        # 3. Save the new/refreshed token
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    service = build('drive', 'v3', credentials=creds)
    return service

def upload_to_drive(file_path, folder_id=None):
    """
    Uploads a file to Google Drive.
    
    :param file_path: The path to the local file to upload.
    :param folder_id: The Google Drive ID of the target folder. None for root.
    :return: The ID of the uploaded file, or None on failure.
    """
    try:
        service = get_drive_service()
        
        # 1. Guess the MIME type from the file extension
        mime_type, _ = mimetypes.guess_type(file_path)
        if not mime_type:
            mime_type = 'application/octet-stream' # Default if type cannot be guessed

        # 2. Set file metadata (name and parent folder)
        file_metadata = {'name': os.path.basename(file_path)}
        if folder_id:
            file_metadata['parents'] = [folder_id]
            
        # 3. Create media object for upload
        media = MediaFileUpload(file_path, mimetype=mime_type, resumable=True)
        
        # 4. Execute the upload
        uploaded = service.files().create(
            body=file_metadata, 
            media_body=media, 
            fields='id'
        ).execute()
        
        file_id = uploaded.get('id')
        print(f"✅ Successfully uploaded '{os.path.basename(file_path)}'. ID: {file_id}")
        return file_id
        
    except FileNotFoundError:
        print(f"❌ Error: The file was not found at path: {file_path}")
        return None
    except Exception as e:
        print(f"❌ An error occurred during upload: {e}")
        return None

def download_from_drive(file_id, local_file_path):
    """
    Downloads a file from Google Drive using its file ID.
    TO FIND THE FILE ID ON GOOGLE DRIVE, SIMPLY GO TO THE GOOGLE DRIVE WEBSITE, CLICK ON THE THING YOU WANT, AND THE LONG STRING OF NUMBERS AND CHARACTERS AFTER "/folders" IN THE WEB ADDRESS IS IT
    :param file_id: The ID of the file to download from Drive.
    :param local_file_path: The path and name to save the file locally.
    :return: True on success, False on failure.
    """
    try:
        service = get_drive_service()

        # 1. Execute the download request using alt='media'
        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)

        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Downloading... {int(status.progress() * 100)}%")

        # ✅ Extract raw bytes from the BytesIO buffer
        file_content = fh.getvalue()

        if isinstance(file_content, dict):
            print("⚠️ Drive returned a dictionary instead of bytes — likely an error:")
            print(file_content)   # <-- this shows the error message from Drive
            return False          # Stop here before writing bad data

        # 2. Write the downloaded byte content to a local file
        with open(local_file_path, 'wb') as f:
            f.write(file_content)

        print(f"⬇️ Successfully downloaded file ID {file_id} to '{local_file_path}'.")
        return True

    except Exception as e:
        print(f"❌ An error occurred during download for file ID {file_id}: {e}", flush=True)
        return False




In [4]:
import os
import cv2
from huggingface_hub import HfFileSystem

# === CONFIG ===
TEMP_DIR = "temp_videos"
os.makedirs(TEMP_DIR, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN")  # set via environment variable
fs = HfFileSystem(token=HF_TOKEN)

def load_video(row):
    """
    Given a DataFrame row (with 'source' and 'filepath'),
    downloads the video if needed and returns the local file path.

    - if source == 'huggingface': downloads via fsspec (fs) into TEMP_DIR
    - if source == 'googledrive': downloads using other_info['fileID'] and the existing
      Google Drive helper functions (download_from_drive)
    """
    source = row["source"]
    filepath = row["filepath"]

    # Always keep the same filename
    filename = os.path.basename(filepath)
    local_path = os.path.join(TEMP_DIR, filename)

    if source == "huggingface":
        # Download if not already present
        if not os.path.exists(local_path):
            print(f"Downloading {filepath} from Hugging Face...")
            with fs.open(filepath, "rb") as src, open(local_path, "wb") as dst:
                dst.write(src.read())
        return os.path.abspath(local_path)

    elif source == "googledrive":
        # Expect 'other_info' to contain the file ID
        try:
            other_info = row["other_info"]
            file_id = other_info["fileID"]
        except Exception:
            raise ValueError("For 'googledrive' source, row['other_info']['fileID'] is required.")

        # Download only if not already saved locally
        if not os.path.exists(local_path):
            print(f"Downloading {filename} from Google Drive (fileID={file_id})...")
            success = download_from_drive(file_id, local_path)
            if not success:
                raise RuntimeError(f"Failed to download file ID {file_id} from Google Drive.")

        return os.path.abspath(local_path)

    else:
        raise NotImplementedError(f"Source '{source}' not supported yet.")

def script1(video_path):
    """
    Simple video player using OpenCV.
    Plays until the end or until 'q' is pressed.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return

    print(f"Playing video: {video_path}")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow("Video", frame)
        if cv2.waitKey(25) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

def delete_video(video_path):
    """
    Deletes a local video file (e.g., one downloaded by load_video()).
    Only allows deletion inside the temp_videos folder for safety.
    """
    abs_path = os.path.abspath(video_path)
    temp_dir_abs = os.path.abspath(TEMP_DIR)

    # Ensure safety: only delete inside temp_videos
    if abs_path.startswith(temp_dir_abs) and os.path.exists(abs_path):
        os.remove(abs_path)
        print(f"Deleted: {abs_path}")
    else:
        print(f"Skipped deletion (unsafe path): {abs_path}")


# === Example usage ===
# import pandas as pd
# df = pd.read_hdf("nexar_train_prepared.h5", key="data")
# row = df.iloc[0]
# local_path = load_video(row)
# script1(local_path)
# delete_video(local_path)


### Example Usage

In [ ]:
#loading up a saved dataframe
import pandas as pd
df = pd.read_hdf("nexar_train_prepared.h5", key="data")
df

,filepath,source,event_time,event_description,split,other_info
0,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.500,collision,train,"{""time_of_alert"": 18.633, ""light_conditions"": ..."
1,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.800,collision,train,"{""time_of_alert"": 19.233, ""light_conditions"": ..."
2,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,20.101,collision,train,"{""time_of_alert"": 20.068, ""light_conditions"": ..."
3,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,19.267,collision,train,"{""time_of_alert"": 18.333, ""light_conditions"": ..."
4,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,21.146,collision,train,"{""time_of_alert"": 18.196, ""light_conditions"": ..."
...,...,...,...,...,...,...
1495,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1496,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1497,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."
1498,hf://datasets/nexar-ai/nexar_collision_predict...,huggingface,NaN,no collision,train,"{""time_of_alert"": NaN, ""light_conditions"": ""No..."


In [ ]:
row = df.iloc[0]
local_path = load_video(row)
script1(local_path) #random function that displays the video clip
delete_video(local_path)

Playing video: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4
Deleted: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4


In [ ]:
import pandas as pd
import concurrent.futures

def script2(video_path):
    """
    Return the length of a video in seconds as a float.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    if fps <= 0 or frame_count <= 0:
        return None
    return float(frame_count / fps)


# === Batch processing ===
input_path = "nexar_train_prepared.h5"
output_path = "output.h5"
df = pd.read_hdf(input_path, key="data")
results = []
n = 10  # number of clips per batch
for i in range(3):  # run 3 batches
    start = i * n
    end = start + n
    if start >= len(df):
        break
    batch = df.iloc[start:end]
    # Load videos concurrently
    with concurrent.futures.ThreadPoolExecutor(max_workers=n) as executor:
        futures = {executor.submit(load_video, row): idx for idx, row in batch.iterrows()}
        local_paths = {}
        for future in concurrent.futures.as_completed(futures):
            idx = futures[future]
            try:
                local_paths[idx] = future.result()
            except Exception as e:
                print(f"Error loading row {idx}: {e}")
    for idx, local_path in local_paths.items():
        clip_length = script2(local_path)
        row_data = batch.loc[idx].to_dict()
        row_data["output"] = clip_length
        results.append(row_data)
        delete_video(local_path)

output_df = pd.DataFrame(results)
output_df.to_hdf(output_path, key="data", mode="w")

print(f"✅ Output saved to {output_path}")
print(output_df.head())



Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00408.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00128.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00072.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00471.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00205.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00822.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00208.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00457.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00900.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00948.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00641.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00841.mp4
Deleted: /Users/aneeshkalla/Desktop/AI_driving_tool/temp_videos/00938.mp4
Deleted: /Users/aneeshkalla/Desktop/A

In [ ]:
# Example: one-row DataFrame for testing
data = [{
    "filepath": "00822.mp4",  # name of the file in Drive
    "source": "googledrive",        # triggers the Drive branch in load_video()
    "event_time": None,
    "event_description": "test_download",
    "split": "demo",
    "other_info": {"fileID": "18ZrK_DsUHZLjJ7qPDDHECaVmcp-fthK8"}  # your file ID
}]
df = pd.DataFrame(data)
# Preview to confirm
print(df)
# ✅ Use your load_video() function
row = df.iloc[0]
local_path = load_video(row)
print("\nDownloaded local file path:", local_path)

    filepath       source event_time event_description split  \
0  00822.mp4  googledrive       None     test_download  demo   

                                        other_info  
0  {'fileID': '18ZrK_DsUHZLjJ7qPDDHECaVmcp-fthK8'}  

Downloaded local file path: /Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4


In [ ]:

uploaded_id = upload_to_drive("/Users/aneeshkalla/Desktop/Driving_Tool/temp_videos/00822.mp4", folder_id="1dtgKrSeaAyv4WrbGVr8zeoKH_7B_7bPg")


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=221407270655-90npb4apkqrkpfmpehtn4hpfb3971kvn.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50266%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=TcPl5Z1UcuktxBBo7j1n8CDaBO7F5v&access_type=offline
✅ Successfully uploaded '00822.mp4'. ID: 1CKFu7T83RZNehEPb-IfxpgZkfo-i1TN6
